# VietMedBridge — 01g: Thu tiếp các URL Stage A còn thiếu

Dùng **runtime CPU mới**, cùng Drive DATA_ROOT. Notebook tự đọc đủ
1.000 ID, kiểm raw checkpoint và ledger sau retry, trích lại 11 browser
captures sẵn có (bao gồm công việc của 01f), giữ 4 ứng viên 01c để review,
rồi xử lý 68 ID còn lại bằng HTTP Chrome TLS → Crawl4AI stealth →
Scrapling stealth. Robots HTTP lỗi được thử đọc bằng browser riêng;
chỉ response robots thật mới được dùng để quyết định tải bài.

Có checkpoint từng ID; không tải lại 917 nguồn gốc hoặc 11 captures.
Đọc robots bằng Chrome TLS, retry lỗi mạng/5xx có giới hạn. Giữ policy
hold cho Disallow đã xác nhận, 401/403/429 hoặc robots chưa đọc được.
Đầu ra là ứng viên review; notebook chưa gộp/sửa corpus Stage A.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "advanced_recovery_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Cài bộ HTTP/browser — CPU, không dùng LLM


In [ ]:
import importlib.metadata
import subprocess

CRAWL4AI_VERSION = "0.9.4"
os.environ["CRAWL4_AI_BASE_DIRECTORY"] = str(WORK_DIR / "crawl4ai")
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    f"crawl4ai=={CRAWL4AI_VERSION}", "scrapling[fetchers]==0.4.15",
    "playwright==1.63.0", "patchright==1.63.0", "playwright-stealth==2.0.3",
    "curl_cffi==0.16.3"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True)
subprocess.run([sys.executable, "-m", "patchright", "install", "chromium"], check=True)
from vietmedbridge.advanced_recovery import AdvancedRecoveryRuntime
from vietmedbridge.stage_a_recovery import recover_remaining

VERSIONS = {p: importlib.metadata.version(p) for p in (
    "crawl4ai", "scrapling", "playwright", "curl_cffi", "patchright", "playwright-stealth",
)}
VERSIONS["crawl4ai_release"] = CRAWL4AI_VERSION
print(VERSIONS)


## 2. Xác minh toàn bộ Stage A và chọn đúng nhóm còn thiếu


In [ ]:
from vietmedbridge.stage_a_recovery import load_remaining_plan
from vietmedbridge.artifacts import atomic_json, publish_file, sha256_file

PLAN, PROVENANCE, SOURCE_FILES = load_remaining_plan(DATA_ROOT, work_dir=WORK_DIR)
if len(PLAN["official_pairs"]) != 1000:
    raise ValueError("Notebook này dành cho mẫu Stage A 1.000 URL đã audit.")
print("Tổng ID:", len(PLAN["official_pairs"]))
print("HTTP gốc:", PLAN["baseline_http_captures"])
print("Các ID sau retry:", PLAN["failed_ids"], "| cần thử tiếp:", PLAN["remaining_ids"])
print("11 bài đã có được trích lại tự động; không cần chạy 01f trước.")


## 3. Thu tiếp nhóm còn thiếu và lưu từng ID

Nếu runtime ngắt, chạy lại cùng notebook/cấu hình; ID đã ghi checkpoint
được kiểm hash và dùng lại. MAX_NEW_IDS chỉ giới hạn công việc trong
phiên hiện tại, không làm rơi ID khỏi ledger. Đổi RECOVERY_LABEL để tạo
lượt retry mới khi cần thử lại lỗi tạm thời; giữ lượt cũ làm bằng chứng.


In [ ]:
RECOVERY_LABEL = "advanced-v1"
MAX_NEW_IDS = None  # None = xử lý đủ nhóm còn thiếu; có thể đặt 10 cho mỗi phiên.
DOMAIN_PROFILES = {}  # Tùy chỉnh selector/resource_hosts theo bằng chứng của từng domain.
REPORT_DIR = DATA_ROOT / "reports/crawl_recovery/stage-a-v2"
async with AdvancedRecoveryRuntime(WORK_DIR / "recovery_browser", profiles=DOMAIN_PROFILES) as engines:
    RESULT = await recover_remaining(
        PLAN, REPORT_DIR / "remaining", engines.get,
        browser_probe=engines.browser_probe, advanced_runtime=engines,
        provenance=PROVENANCE, versions=VERSIONS,
        label=RECOVERY_LABEL, max_new_ids=MAX_NEW_IDS,
    )
RESULT_DIR = Path(RESULT["output_dir"])
print(json.dumps({k: RESULT[k] for k in (
    "official_ids", "baseline_http_captures", "remaining_input_ids",
    "processed_remaining_ids", "states", "canonical_corpus_modified",
)}, ensure_ascii=False, indent=2))


## 4. Xuất ledger đủ 1.000 ID và gói bằng chứng để gửi kiểm tra


In [ ]:
import shutil
import pandas as pd
from urllib.parse import urlsplit
from google.colab import files

for name, source in SOURCE_FILES.items():
    name = "official_stage_a.parquet" if name == "official_stage_a" else name
    publish_file(source, RESULT_DIR / "inputs" / name)
pd.DataFrame(RESULT["coverage_ledger"]).to_csv(
    RESULT_DIR / "coverage_1000.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(RESULT["records"]).to_csv(
    RESULT_DIR / "recovery_results.csv", index=False, encoding="utf-8-sig")
overview = pd.DataFrame(RESULT["coverage_ledger"])
overview["domain"] = overview["url"].map(lambda value: urlsplit(value).hostname)
overview.groupby(["domain", "state"]).size().rename("ids").reset_index().to_csv(
    RESULT_DIR / "coverage_by_domain.csv", index=False, encoding="utf-8-sig")
captured_states = {"BASELINE_HTTP_CAPTURE", "CACHED_BROWSER_REVIEW",
                   "LEGACY_CANDIDATE_REVIEW", "ARTICLE_CANDIDATE_REVIEW"}
overview[~overview["state"].isin(captured_states)].to_csv(
    RESULT_DIR / "unresolved_urls.csv", index=False, encoding="utf-8-sig")
artifacts = {p.relative_to(RESULT_DIR).as_posix(): sha256_file(p)
             for p in RESULT_DIR.rglob("*") if p.is_file() and p.name != "export_hashes.json"}
atomic_json(RESULT_DIR / "export_hashes.json", artifacts)
display(pd.DataFrame(RESULT["records"])[["doc_id", "url", "state"]])
local_zip = Path(shutil.make_archive(
    str(WORK_DIR / f"remaining-stage-a-{RESULT['experiment']}"), "zip", root_dir=RESULT_DIR))
publish_file(local_zip, REPORT_DIR / local_zip.name)
print("Gửi ZIP này để kiểm đủ URL và nội dung:", local_zip.name)
files.download(str(local_zip))
